[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/raya-lucaria/ia_o26/blob/main/course/10_deep_learning/_assets/04_finetuning_llm.ipynb)

# Notebook 4 · Fine-tuning de un LLM con LoRA: tu dataset

Vas a adaptar **Qwen2.5-Instruct** a tu tarea de texto con **LoRA**. Compararás las respuestas antes y después del fine-tuning, verás qué cambió en el modelo y descargarás tu adaptador.

1. En Colab, abre «Entorno de ejecución → Cambiar tipo de entorno», elige **T4 GPU** y guarda.
2. Arrastra tu `texto.zip` al panel **📁 Archivos** de la izquierda y espera a que termine la carga. Si no tienes un zip, el cuaderno usa un demo de preguntas que se responden con SQL.
3. Selecciona «Entorno de ejecución → **Ejecutar todo**». Si vas a probar ajustes, ejecuta hasta la sección 07 y reserva la 08 para tu configuración final.

Usa `MODO = "corto"` en clase. El modo `"largo"` usa un modelo más grande, ajusta también el MLP y dedica más pasos al entrenamiento. La duración depende del entorno, las descargas y los datos.

| Sección | Qué haces |
|---|---|
| 01 · Configuración | Eliges el modo y compruebas la GPU. |
| 02 · Tu zip | Revisas los datos y los repartes en tres splits. |
| 03 · Antes | Lees las respuestas del modelo base. |
| 04 · LoRA por dentro | Descompones la actualización y cuentas sus parámetros. |
| 05 · Entrenamiento | Ajustas el adaptador con loss solo en el completion. |
| 06 · Inspección | Mides el cambio por capa, ajustas su escala y pruebas el merge. |
| 07 · Métricas | Comparas base y LoRA en **validation**. |
| 08 · Testeo | Evalúas en **test una sola vez** y descargas el adaptador. |

## 01 · Instalar y configurar

Elige `MODO` en la primera línea de código. `LARGO` indica si activaste el modo largo. La celda instala los paquetes que faltan y conserva las versiones ya instaladas.

| Paquete | Para qué lo usamos |
|---|---|
| `transformers` | Cargar el LLM, el tokenizer y su plantilla de chat. |
| `peft` | Añadir, activar, desactivar y guardar LoRA. |
| `accelerate` | Apoyar la carga del modelo. |
| `safetensors` | Guardar los pesos del adaptador. |

In [ ]:
MODO = "corto"   # "corto" (≈ 20–25 min en T4) o "largo" (≈ 45–60 min)
LARGO = MODO == "largo"

# Instala solo lo que falta; no actualiza lo que Colab ya trae.
import importlib.util
import subprocess
import sys

FALTAN = [p for p in ["transformers", "peft", "accelerate", "safetensors"]
          if importlib.util.find_spec(p) is None]
if FALTAN:
    print("Instalando:", ", ".join(FALTAN))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *FALTAN],
                   check=True)
else:
    print("No falta nada que instalar.")
print("MODO =", MODO, "· LARGO =", LARGO)

- `DEVICE` indica dónde se ejecuta el modelo. Si Colab avisa que estás en CPU, activa la **T4 GPU** y vuelve a ejecutar todo.
- `DTYPE` indica la precisión de los pesos base. En GPU se cargan en fp16, que ocupa la mitad por parámetro que fp32. En CPU se usa fp32.
- `RAPIDO=1` fuerza una prueba en CPU con un Qwen diminuto. Sus respuestas y métricas sirven para revisar el flujo.

In [ ]:
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import peft
import transformers
import huggingface_hub

# RAPIDO=1: prueba local en CPU con un Qwen diminuto (segundos).
RAPIDO = os.environ.get("RAPIDO") == "1"
HAY_GPU = torch.cuda.is_available()
DEVICE = torch.device("cuda" if HAY_GPU and not RAPIDO else "cpu")
CUDA = DEVICE.type == "cuda"
# fp16 en la GPU (la T4 no tiene bf16); fp32 en CPU.
DTYPE = torch.float16 if CUDA else torch.float32

if not HAY_GPU and not RAPIDO:
    print("=" * 72)
    print("  ¡AVISO! ESTE ENTORNO NO TIENE GPU.")
    print("  Entrenar un LLM en CPU tardaría horas. En Colab:")
    print("    Entorno de ejecución → Cambiar tipo de entorno → T4 GPU")
    print("    → Guardar, y luego Entorno de ejecución → Ejecutar todo.")
    print("=" * 72)

for nombre, mod in [("torch", torch), ("transformers", transformers),
                    ("peft", peft), ("huggingface_hub", huggingface_hub)]:
    print(f"{nombre:16s}{mod.__version__}")
print("DEVICE =", DEVICE, "· DTYPE =", DTYPE, "· RAPIDO =", RAPIDO)
if HAY_GPU:
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} · {props.total_memory / 2**30:.1f} GiB")

| Modo | Modelo base | Dónde se aplica LoRA |
|---|---|---|
| Corto | `Qwen/Qwen2.5-0.5B-Instruct` | Proyecciones de atención. |
| Largo | `Qwen/Qwen2.5-1.5B-Instruct` | Proyecciones de atención y del MLP. |

Un modelo *Instruct* ya está ajustado para seguir instrucciones. Aquí lo adaptas a la forma de responder que muestran tus ejemplos. El modo largo requiere más memoria. La prueba de merge crea además una copia del modelo.

La celda muestra `P`, la configuración efectiva de tu corrida.

| Claves de `P` | Qué controlan |
|---|---|
| `n_demo` | Tamaño de la muestra del demo. |
| `max_len`, `max_new` | Límite de tokens por ejemplo al preparar la loss y límite de tokens nuevos al generar. |
| `batch`, `acum` | Ejemplos por batch y batches cuyos gradientes se acumulan antes de actualizar. |
| `pasos`, `lr`, `val_cada` | Actualizaciones programadas, learning rate y frecuencia de evaluación en validation. |
| `n_val_loss`, `n_eval` | Topes de ejemplos por split para medir la loss y evaluar las respuestas generadas. |
| `rango`, `alfa`, `modulos` | Rango, escala y capas del adaptador. |

`semilla(s)` fija el azar de Python, NumPy y PyTorch. En la gráfica de entrenamiento, azul es train y naranja es validation.

In [ ]:
# El modelo base. Los dos Qwen2.5-Instruct son Apache-2.0.
if RAPIDO:
    MODELO = "trl-internal-testing/tiny-Qwen2ForCausalLM-2.5"
elif LARGO:
    MODELO = "Qwen/Qwen2.5-1.5B-Instruct"
else:
    MODELO = "Qwen/Qwen2.5-0.5B-Instruct"
DEMO = "b-mc2/sql-create-context"     # el dataset del demo (CC BY 4.0)

# Tamaños por modo. pasos = actualizaciones del adaptador; cada paso ve
# batch × acum ejemplos.
ATENCION = ["q_proj", "k_proj", "v_proj", "o_proj"]
if RAPIDO:
    P = dict(n_demo=200, max_len=128, batch=2, acum=1, pasos=6, lr=2e-4,
             val_cada=3, n_val_loss=8, n_eval=4, max_new=16,
             rango=8, alfa=16, modulos=ATENCION)
elif LARGO:
    P = dict(n_demo=3000, max_len=512, batch=4, acum=2, pasos=800, lr=2e-4,
             val_cada=50, n_val_loss=200, n_eval=150, max_new=192,
             rango=8, alfa=16, modulos=ATENCION)
else:
    P = dict(n_demo=1000, max_len=512, batch=4, acum=2, pasos=300, lr=2e-4,
             val_cada=25, n_val_loss=100, n_eval=100, max_new=128,
             rango=8, alfa=16, modulos=ATENCION)
if LARGO:                               # ➕ LoRA también en el MLP
    P["modulos"] = ATENCION + ["gate_proj", "up_proj", "down_proj"]
display(pd.Series({k: str(v) for k, v in P.items()}, name="valor").to_frame())

# Un color fijo por conjunto, igual que en los notebooks 2 y 3.
COLORES = {"train": "#1f77b4", "validation": "#ff7f0e", "test": "#2ca02c"}


def semilla(s: int):
    """Fija las semillas de random, NumPy y PyTorch (CPU y GPU)."""
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)        # también siembra la GPU


def mostrar(df):
    """Una tabla de texto: celdas alineadas a la izquierda y con saltos."""
    try:
        display(df.style.set_properties(
            **{"text-align": "left", "white-space": "pre-wrap"}))
    except Exception:           # sin jinja2 o fuera de Jupyter
        with pd.option_context("display.max_colwidth", 80):
            display(df)


semilla(0)
plt.style.use("default")
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3,
                     "figure.dpi": 80, "savefig.dpi": 80,
                     "axes.spines.top": False, "axes.spines.right": False})
try:                            # en Colab: figuras en JPEG, el notebook pesa menos
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("jpeg", pil_kwargs={"quality": 80})
except Exception:
    pass

## 02 · Tu zip

Empaqueta `datos.jsonl`, `prompts.txt` y `LEEME.md` en la raíz de `texto.zip`. Cada línea de `datos.jsonl` lleva un `prompt`, la entrada al modelo, y un `completion`, la respuesta esperada.

| Dónde está tu zip | Qué haces |
|---|---|
| En tu computadora | Arrástralo al panel **📁 Archivos** de Colab y espera a que termine la carga. |
| En Google Drive | Descomenta las líneas de Drive y ajusta la ruta en `RUTA_ZIP`. |
| Si falla el arrastre | Descomenta la línea con `files.upload()` para abrir el selector de archivos. La ejecución espera a que elijas el zip. |

Con `RUTA_ZIP = ""`, el cuaderno busca primero `texto.zip` en `/content` y en la carpeta actual. También reconoce otros nombres si el zip trae `datos.jsonl` en la raíz. Si tienes varios zips, indica cuál usar en `RUTA_ZIP`. Si encuentra un zip de imágenes como entrada, te dirige al notebook 3.

Si dejas la ruta vacía y la búsqueda no encuentra un zip de texto ni de imágenes, la celda crea `texto_demo.zip` con una muestra de `b-mc2/sql-create-context`. Cada entrada trae el esquema de las tablas y una pregunta. La respuesta esperada es una consulta SQL.

In [ ]:
RUTA_ZIP = ""    # vacío: busca texto.zip; si no hay, arma el demo
# Si lo subiste a Drive, quita los # de estas dos líneas:
# from google.colab import drive; drive.mount("/content/drive")
# RUTA_ZIP = "/content/drive/MyDrive/texto.zip"
# O súbelo con un botón (detiene «Ejecutar todo» hasta que elijas):
# from google.colab import files; RUTA_ZIP = list(files.upload())[0]

import glob
import json
import zipfile

NB3 = ("https://colab.research.google.com/github/raya-lucaria/ia_o26/blob/"
       "main/course/10_deep_learning/_assets/03_finetuning_lora.ipynb")


def tipo_de_zip(ruta):
    """'texto', 'imagenes' o None, según lo que traiga en la raíz."""
    try:
        nombres = set(zipfile.ZipFile(ruta).namelist())
    except (zipfile.BadZipFile, OSError):
        return None
    if "datos.jsonl" in nombres:
        return "texto"
    return "imagenes" if "metadata.csv" in nombres else None


def buscar_zip():
    """El zip de texto que subiste: texto.zip, o cualquier .zip con
    datos.jsonl, en /content o en la carpeta actual."""
    carpetas = list(dict.fromkeys(["/content", os.getcwd()]))
    for c in carpetas:
        if os.path.exists(os.path.join(c, "texto.zip")):
            return os.path.join(c, "texto.zip")
    zips = [z for c in carpetas for z in sorted(glob.glob(f"{c}/*.zip"))
            if not z.endswith("texto_demo.zip")]
    for z in zips:
        if tipo_de_zip(z) == "texto":
            return z
    imagenes = [z for z in zips if tipo_de_zip(z) == "imagenes"]
    if imagenes:
        raise ValueError(
            f"{imagenes[0]} es un zip de IMÁGENES (trae metadata.csv).\n"
            f"Este notebook es de texto; las imágenes van en el notebook 3:\n"
            f"{NB3}")
    return None

In [ ]:
def armar_demo(ruta, n):
    """Arma un texto.zip de demostración, en el formato de la tarea, con n
    ejemplos de b-mc2/sql-create-context (pregunta + tablas → SQL)."""
    from huggingface_hub import hf_hub_download
    archivo = hf_hub_download(DEMO, "sql_create_context_v4.json",
                              repo_type="dataset")
    with open(archivo, encoding="utf-8") as f:
        filas = random.Random(0).sample(json.load(f), n)
    datos = [{"prompt": f"Tablas: {e['context']}\nPregunta: {e['question']}",
              "completion": e["answer"]} for e in filas]
    # Cuatro prompts que no están en el dataset; «\n» es un salto de línea.
    prompts = [
        r"Tablas: CREATE TABLE students (name VARCHAR, age INTEGER)\n"
        r"Pregunta: How many students are older than 20?",
        r"Tablas: CREATE TABLE courses (id INTEGER, title VARCHAR, "
        r"credits INTEGER)\nPregunta: What is the title of course 7?",
        r"Tablas: CREATE TABLE songs (title VARCHAR, artist VARCHAR, "
        r"year INTEGER)\nPregunta: Which artists released songs in 1999?",
        r"Tablas: CREATE TABLE cities (name VARCHAR, country VARCHAR, "
        r"population INTEGER)\nPregunta: What is the total population of "
        r"cities in Mexico?"]
    leeme = (
        "# Mi dataset\n"
        "- **Qué quiero que aprenda el modelo:** escribir la consulta SQL "
        "que responde una pregunta sobre unas tablas\n"
        "- **Ruta:** texto\n"
        f"- **Fuente:** https://huggingface.co/datasets/{DEMO}\n"
        "- **Licencia:** CC BY 4.0\n"
        f"- **Cuántos ejemplos y cómo los elegí:** {n} al azar con semilla 0\n")
    with zipfile.ZipFile(ruta, "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr("datos.jsonl", "".join(
            json.dumps(e, ensure_ascii=False) + "\n" for e in datos))
        z.writestr("prompts.txt", "\n".join(prompts) + "\n")
        z.writestr("LEEME.md", leeme)
    return ruta


if RUTA_ZIP:
    assert os.path.exists(RUTA_ZIP), f"no existe {RUTA_ZIP}: revisa la ruta"
    if tipo_de_zip(RUTA_ZIP) == "imagenes":
        raise ValueError(f"{RUTA_ZIP} es un zip de IMÁGENES: úsalo en el "
                         f"notebook 3:\n{NB3}")
else:
    RUTA_ZIP = buscar_zip()
    if RUTA_ZIP is None:
        print("No encontré tu texto.zip: armo el demo (texto → SQL).")
        RUTA_ZIP = armar_demo(os.path.join(os.getcwd(), "texto_demo.zip"),
                              P["n_demo"])
ES_DEMO = RUTA_ZIP.endswith("texto_demo.zip")
print("Uso:", RUTA_ZIP, "(demo)" if ES_DEMO else "(tu zip)")

`revisar_zip` es la función de la tarea. Comprueba los archivos obligatorios, la cantidad de prompts y ejemplos, las llaves de cada ejemplo y que haya un completion. Si falla, corrige lo que indique el mensaje, vuelve a subir el zip y ejecuta la sección 02.

La celda lee la ficha, muestra ejemplos y resume la longitud de prompts y completions en **caracteres**. Úsala para detectar entradas largas. El recorte en la sección 05 se decide en **tokens** y también cuenta las marcas de la plantilla de chat.

En `prompts.txt`, escribe un prompt por línea. Si un prompt necesita un salto de línea interno, escribe los dos caracteres `\n`. Al leer el archivo, la celda los convierte en un salto de línea.

In [ ]:
# La función de la tarea, copiada literal: dice OK o qué está mal.
import csv, io, json, zipfile
from PIL import Image

def revisar_zip(ruta):
    """Dice si el zip sigue el formato de la tarea. No cambia nada."""
    z = zipfile.ZipFile(ruta)
    nombres = set(z.namelist())
    faltan = {"prompts.txt", "LEEME.md"} - nombres
    assert not faltan, f"faltan en la raíz del zip: {sorted(faltan)}"
    prompts = z.read("prompts.txt").decode("utf-8").split("\n")
    prompts = [p for p in prompts if p.strip()]
    assert 3 <= len(prompts) <= 5, f"prompts.txt: {len(prompts)} prompts, van de 3 a 5"
    if "metadata.csv" in nombres:                     # imágenes
        filas = list(csv.DictReader(io.StringIO(z.read("metadata.csv").decode("utf-8"))))
        assert filas and set(filas[0]) == {"file_name", "text"}, "columnas: file_name,text"
        assert len(filas) >= 50, f"solo {len(filas)} imágenes; mínimo 50"
        for fila in filas:
            assert fila["file_name"] in nombres, f"no está {fila['file_name']}"
            assert fila["text"].strip(), f"{fila['file_name']} sin texto"
            Image.open(z.open(fila["file_name"])).verify()
        print(f"OK · imágenes: {len(filas)} · prompts: {len(prompts)}")
    elif "datos.jsonl" in nombres:                    # texto
        lineas = z.read("datos.jsonl").decode("utf-8").splitlines()
        ejemplos = [json.loads(l) for l in lineas if l.strip()]
        assert len(ejemplos) >= 200, f"solo {len(ejemplos)} ejemplos; mínimo 200"
        for i, ej in enumerate(ejemplos, 1):
            assert set(ej) == {"prompt", "completion"}, f"línea {i}: llaves {sorted(ej)}"
            assert ej["completion"].strip(), f"línea {i}: completion vacío"
        print(f"OK · ejemplos: {len(ejemplos)} · prompts: {len(prompts)}")
    else:
        raise AssertionError("falta metadata.csv (imágenes) o datos.jsonl (texto)")

revisar_zip(RUTA_ZIP)

In [ ]:
_z = zipfile.ZipFile(RUTA_ZIP)
EJEMPLOS = [json.loads(l) for l in
            _z.read("datos.jsonl").decode("utf-8").splitlines() if l.strip()]
# En prompts.txt, «\n» escrito con dos caracteres es un salto de línea.
PROMPTS = [p.replace("\\n", "\n").strip() for p in
           _z.read("prompts.txt").decode("utf-8").split("\n") if p.strip()]
LEEME = _z.read("LEEME.md").decode("utf-8")
try:
    from IPython.display import Markdown
    display(Markdown(LEEME))
except ImportError:
    print(LEEME)

largos = pd.DataFrame({
    "prompt": [len(e["prompt"]) for e in EJEMPLOS],
    "completion": [len(e["completion"]) for e in EJEMPLOS]})
print(f"{len(EJEMPLOS)} ejemplos · {len(PROMPTS)} prompts de prueba")
display(largos.describe().loc[["mean", "50%", "max"]].round(0)
        .rename(index={"mean": "promedio (caracteres)",
                       "50%": "mediana", "max": "máximo"}))


def recorta(texto, n=300):
    return texto if len(texto) <= n else texto[:n] + " …"


muestra = random.Random(1).sample(EJEMPLOS, 3)
mostrar(pd.DataFrame([{"prompt": recorta(e["prompt"]),
                       "completion": recorta(e["completion"])}
                      for e in muestra]))

| Split | Proporción aproximada | Para qué se usa |
|---|---:|---|
| **Train** | 80 % | Calcular los gradientes y actualizar el adaptador. |
| **Validation** | 10 % | Seguir las curvas, comparar respuestas y decidir ajustes. |
| **Test** | 10 % | Medir el resultado final **una sola vez**, en la sección 08. |

El reparto baraja las filas con semilla fija. Con el mismo archivo y orden de filas, se repite el mismo split. Revisa los duplicados antes de empaquetar, porque dos filas iguales pueden quedar en splits distintos.

In [ ]:
# 80 / 10 / 10 con semilla fija: siempre los mismos ejemplos en cada parte.
orden = np.random.default_rng(0).permutation(len(EJEMPLOS))
n_10 = max(1, round(0.1 * len(EJEMPLOS)))
DATOS = {"validation": [EJEMPLOS[i] for i in orden[:n_10]],
         "test": [EJEMPLOS[i] for i in orden[n_10:2 * n_10]],
         "train": [EJEMPLOS[i] for i in orden[2 * n_10:]]}
display(pd.Series({k: len(v) for k, v in DATOS.items()},
                  name="ejemplos").to_frame())

## 03 · Antes: qué contesta el modelo base

Primero genera con Qwen tal como viene. La celda carga el tokenizer y el modelo, imprime cuántos parámetros tiene y congela sus pesos. Las respuestas del base te servirán de referencia para evaluar qué aporta LoRA.

In [ ]:
import contextlib
import gc
import math
import time

from packaging.version import Version
from transformers import AutoModelForCausalLM, AutoTokenizer

# transformers ≥ 4.56 llama «dtype» a lo que antes era «torch_dtype».
_DT = ("dtype" if Version(transformers.__version__) >= Version("4.56")
       else "torch_dtype")
TOK = AutoTokenizer.from_pretrained(MODELO)
if TOK.pad_token is None:
    TOK.pad_token = TOK.eos_token
LM = AutoModelForCausalLM.from_pretrained(MODELO, **{_DT: DTYPE}).to(DEVICE)
LM.requires_grad_(False)                 # congelado: solo LoRA va a entrenar
LM.eval()
# Generación greedy: sin azar ni penalizaciones (Qwen trae sampling por omisión).
_g = LM.generation_config
_g.do_sample, _g.temperature, _g.top_p, _g.top_k = False, None, None, None
_g.repetition_penalty = None
n_lm = sum(p.numel() for p in LM.parameters())
print(f"{MODELO}: {n_lm / 1e6:,.0f} M parámetros · "
      f"{n_lm * LM.dtype.itemsize / 1e9:.2f} GB en {LM.dtype}")


def liberar():
    """Suelta lo que ya no se usa de la memoria (GPU y RAM)."""
    gc.collect()
    if CUDA:
        torch.cuda.empty_cache()

`formatear(prompt)` usa `apply_chat_template` para convertir tu entrada en una conversación. La celda imprime el primer prompt ya formateado. Busca el mensaje de usuario y la marca que abre el turno del asistente. En entrenamiento, el completion ocupa ese turno. Al generar, el modelo lo escribe token a token a partir del mismo prefijo.

In [ ]:
def formatear(prompt):
    """El prompt dentro de la plantilla de chat del modelo, listo para que
    el asistente conteste."""
    return TOK.apply_chat_template([{"role": "user", "content": prompt}],
                                   tokenize=False, add_generation_prompt=True)


print(formatear(PROMPTS[0]))

- **Greedy decoding.** En cada paso, `generar` elige el token más probable. La comparación entre base y LoRA usa la misma entrada y prescinde del azar del *sampling*.
- **Padding.** Al generar por batches, el relleno va a la izquierda y queda enmascarado. La celda decodifica solo los tokens nuevos y omite los tokens especiales.
- **Límite.** `P["max_new"]` acota los tokens nuevos por respuesta. Si una frase queda a medias, revisa ese límite.
- **Base y LoRA.** Una vez instalado el adaptador, `lora=False` lo desactiva temporalmente. Puedes comparar las dos versiones con el mismo modelo cargado.

In [ ]:
def _sin_adaptador(modelo, apagar):
    """Contexto que apaga LoRA (si el modelo ya lo tiene y apagar=True)."""
    if apagar and hasattr(modelo, "disable_adapter"):
        return modelo.disable_adapter()
    return contextlib.nullcontext()


@torch.no_grad()
def generar(prompts, lora=True, max_new=None, modelo=None, lote=16):
    """Respuestas greedy, una por prompt. lora=False → el modelo base."""
    prompts = [prompts] if isinstance(prompts, str) else list(prompts)
    modelo = LM if modelo is None else modelo
    cfg = dict(max_new_tokens=max_new or P["max_new"],
               pad_token_id=TOK.pad_token_id)
    TOK.padding_side = "left"            # al generar, el relleno va a la izquierda
    salida = []
    with _sin_adaptador(modelo, not lora):
        modelo.eval()
        for i in range(0, len(prompts), lote):
            x = TOK([formatear(p) for p in prompts[i:i + lote]],
                    return_tensors="pt", padding=True,
                    add_special_tokens=False).to(DEVICE)
            y = modelo.generate(**x, **cfg)
            salida += TOK.batch_decode(y[:, x["input_ids"].shape[1]:],
                                       skip_special_tokens=True)
    TOK.padding_side = "right"           # al entrenar, a la derecha
    return [s.strip() for s in salida]


def respuestas(prompts, **columnas):
    """Tabla prompt → una columna por modelo (o por lo que pases)."""
    df = pd.DataFrame({"prompt": [recorta(p, 200) for p in prompts]})
    for nombre, textos in columnas.items():
        df[nombre] = [recorta(t, 400) for t in textos]
    return df

In [ ]:
# El modelo base con tus prompts de prueba (los que NO están en el dataset).
t0 = time.perf_counter()
BASE_PROMPTS = generar(PROMPTS, lora=False)
mostrar(respuestas(PROMPTS, base=BASE_PROMPTS))

# Y tres ejemplos de validation, con la respuesta que pide tu dataset.
VAL3 = DATOS["validation"][:3]
mostrar(respuestas([e["prompt"] for e in VAL3],
                   esperado=[e["completion"] for e in VAL3],
                   base=generar([e["prompt"] for e in VAL3], lora=False)))
print(f"({time.perf_counter() - t0:.0f} s generando)")

**Cómo leer las tablas.** La primera reúne los prompts de `prompts.txt` y guarda las respuestas del base para la sección 08. La segunda compara `base` con `esperado` en tres ejemplos de validation. Fíjate por separado en el contenido y en el formato. Una consulta SQL puede resolver bien la pregunta y venir envuelta en una explicación que la referencia no trae.

## 04 · LoRA por dentro

LoRA (*Low-Rank Adaptation*) congela los pesos base y aprende una corrección en las capas lineales elegidas. En esta sección instalas el adaptador, cuentas sus parámetros y compruebas cómo se comporta antes del primer paso.

$$
W=W_0+\Delta W,\qquad \Delta W=\frac{\alpha}{r}BA.
$$

| Símbolo | Forma | Papel |
|---|---|---|
| $W_0$ | $d\times k$ | Matriz congelada de una capa con $k$ entradas y $d$ salidas. |
| $A$ | $r\times k$ | Proyección entrenable de la entrada a $r$ coordenadas. |
| $B$ | $d\times r$ | Proyección entrenable de esas coordenadas a la salida. |
| $r$, $\alpha$ | Escalares | $r$ limita el rango de la corrección y $\alpha/r$ fija su escala. |

- **Forward.** Para una entrada $x$, la parte lineal es $W_0x+\frac{\alpha}{r}B(Ax)$. Las dos proyecciones calculan la corrección sin construir la matriz completa $BA$. Durante el entrenamiento se aplica *dropout* a la entrada de la rama LoRA.
- **Inicio.** $A$ se inicializa al azar y $B=0$. Así $\Delta W=0$ y la capa empieza con la misma salida que la original.
- **Rango y parámetros.** La corrección pasa por $r$ coordenadas, así que $\operatorname{rank}(\Delta W)\le r$. Cuando $r$ es pequeño frente a $d$ y $k$, bastan $r(d+k)$ parámetros para el adaptador frente a los $dk$ de la matriz completa.

| Proyecciones | Qué hacen |
|---|---|
| `q_proj`, `k_proj`, `v_proj` | Construyen las *queries*, *keys* y *values* de la atención. |
| `o_proj` | Proyecta la salida de las cabezas de atención al ancho del modelo. |
| `gate_proj`, `up_proj`, `down_proj` | Transforman la representación de cada token en el MLP. Reciben LoRA en modo largo. |

In [ ]:
import tempfile

from peft import LoraConfig, PeftModel, get_peft_model
from peft.tuners.lora import LoraLayer

CONFIG_LORA = LoraConfig(r=P["rango"], lora_alpha=P["alfa"],
                         target_modules=P["modulos"], lora_dropout=0.05,
                         bias="none", task_type="CAUSAL_LM")


def lora_nuevo():
    """Quita el adaptador que hubiera y pone uno nuevo: A al azar, B = 0."""
    global LM
    if isinstance(LM, PeftModel):
        LM = LM.unload()                 # el modelo base, sin capas LoRA
    semilla(0)
    LM = get_peft_model(LM, CONFIG_LORA)
    for p in LM.parameters():            # adaptador en fp32 (ver §05)
        if p.requires_grad:
            p.data = p.data.float()
    return LM


def capas_lora():
    """{nombre: capa} de cada lineal que lleva LoRA."""
    return {n: m for n, m in LM.named_modules() if isinstance(m, LoraLayer)}


lora_nuevo()
capas = capas_lora()
print(f"{len(capas)} lineales con LoRA, por ejemplo:")
for n in list(capas)[:len(P["modulos"])]:
    print("  ", n)

In [ ]:
# ¿Cuánto entrenas de verdad? Parámetros, % y MB, medidos en el modelo.
total = sum(p.numel() for p in LM.parameters())
entrenables = sum(p.numel() for p in LM.parameters() if p.requires_grad)
mb_modelo = sum(p.numel() * p.element_size() for p in LM.parameters()
                if not p.requires_grad) / 1e6
with tempfile.TemporaryDirectory() as d:  # el adaptador, guardado en disco
    LM.save_pretrained(d)
    mb_adaptador = sum(os.path.getsize(os.path.join(d, f))
                       for f in os.listdir(d)) / 1e6
display(pd.DataFrame({"valor": {
    "parámetros del modelo": f"{total - entrenables:,}",
    "parámetros entrenables (A y B)": f"{entrenables:,}",
    "% entrenable": f"{100 * entrenables / total:.3f} %",
    "MB del modelo (congelado)": f"{mb_modelo:,.0f}",
    "MB del adaptador en disco": f"{mb_adaptador:,.1f}"}}))

# Por tipo de capa, en la primera capa del modelo: d·k contra r(d + k).
filas = []
for n, m in capas.items():
    if ".layers.0." not in n:
        continue
    d, k = m.base_layer.weight.shape
    filas.append({"capa": n.split(".")[-1], "d (salida)": d, "k (entrada)": k,
                  "d·k": d * k, "r(d+k)": P["rango"] * (d + k),
                  "%": round(100 * P["rango"] * (d + k) / (d * k), 2)})
display(pd.DataFrame(filas).set_index("capa"))

In [ ]:
# Con B = 0, ΔW = 0: el modelo con LoRA da EXACTAMENTE lo mismo que el base.
x = TOK([formatear(PROMPTS[0])], return_tensors="pt",
        add_special_tokens=False).to(DEVICE)
with torch.no_grad():
    con = LM(**x).logits
    with LM.disable_adapter():
        sin = LM(**x).logits
print("max |logits con LoRA − logits base| =", (con - sin).abs().max().item())

**Cómo leer las tablas.** El porcentaje global divide los parámetros de $A$ y $B$ entre todos los parámetros del modelo con LoRA. En la tabla del primer bloque, cada porcentaje compara $r(d+k)$ con $dk$ para esa proyección. Solo los parámetros entrenables necesitan estados del optimizador. La memoria de entrenamiento también incluye las activaciones.

Compara los MB de los pesos congelados con el tamaño del adaptador guardado en disco. Por último, lee la diferencia máxima entre los logits con el adaptador recién inicializado y con el adaptador apagado. Con $B=0$, se espera que sea cero.

## 05 · Entrenar

**La loss solo cuenta en el completion.** El prompt le dice al modelo qué responder. Al dejarlo como contexto y asignarle peso cero en la loss, entrenamos al modelo para que produzca la respuesta dada la pregunta. Si también puntuáramos los tokens del prompt, parte de la señal premiaría predecir la pregunta y las marcas de chat. Un prompt largo podría dominar la loss.

| Campo | Qué contiene |
|---|---|
| `ids` | Tokens del prompt formateado, seguidos del completion y su marca de fin. Todo entra al modelo hasta el límite de contexto de esta preparación. |
| `labels` | `-100` para el prefijo de chat y el id de cada token del completion que se conserva. La marca de fin también cuenta si cabe, así el modelo aprende cuándo terminar. |
| `cortado` | Indica si el ejemplo superó `P["max_len"]` y perdió tokens al final. |

La tabla cuenta los recortes por split. Si el prompt llena el espacio y no queda ningún token del completion, el ejemplo se descarta para el cálculo de la loss.

In [ ]:
import html

import torch.nn.functional as F
from transformers import get_cosine_schedule_with_warmup


def tokenizar(ej):
    """ids del prompt + completion y labels con -100 en el prompt."""
    p_txt = formatear(ej["prompt"])
    completo = TOK.apply_chat_template(
        [{"role": "user", "content": ej["prompt"]},
         {"role": "assistant", "content": ej["completion"]}], tokenize=False)
    if completo.startswith(p_txt):       # «completion<|im_end|>»
        c_txt = completo[len(p_txt):].rstrip("\n")
    else:                                # plantilla rara: a mano
        c_txt = ej["completion"] + TOK.eos_token
    p = TOK(p_txt, add_special_tokens=False)["input_ids"]
    c = TOK(c_txt, add_special_tokens=False)["input_ids"]
    return {"ids": (p + c)[:P["max_len"]],
            "labels": ([-100] * len(p) + c)[:P["max_len"]],
            "n_prompt": len(p), "cortado": len(p) + len(c) > P["max_len"]}


TOKS, resumen = {}, {}
for parte, ejs in DATOS.items():
    todos = [tokenizar(e) for e in ejs]
    # sin ningún token de completion (el prompt solo ya llena max_len): fuera
    TOKS[parte] = [t for t in todos if any(l != -100 for l in t["labels"])]
    resumen[parte] = {"ejemplos": len(todos),
                      "cortados": sum(t["cortado"] for t in todos),
                      "descartados": len(todos) - len(TOKS[parte]),
                      "tokens (mediana)": int(np.median(
                          [len(t["ids"]) for t in todos]))}
display(pd.DataFrame(resumen).T)
assert TOKS["train"], "ningún ejemplo de train cabe en max_len: acorta prompts"

**Cómo leer la máscara.** La celda muestra un tramo de un ejemplo de train. En gris están el prompt y las marcas del prefijo, con label `-100`. En verde están los tokens que sirven de objetivo. La tabla amplía la frontera entre ambos.

Un modelo causal usa los logits de la posición anterior para predecir $x_i$. Después de alinear logits y labels, `cross_entropy` ignora el label `-100`. Si $C$ reúne las posiciones del completion que cuentan en el batch, la loss es

$$
\mathcal L=\frac{1}{\lvert C\rvert}\sum_{i\in C}-\log p_\theta(x_i\mid x_{<i}).
$$

El prefijo sigue en $x_{<i}$ y condiciona la respuesta. El promedio da el mismo peso a cada token evaluado, de modo que los completions largos aportan más términos.

In [ ]:
def ver_mascara(t, antes=40, despues=60):
    """El ejemplo token por token: gris = prompt (label -100, no cuenta en la
    loss), verde = completion (sí cuenta)."""
    i0 = max(0, t["n_prompt"] - antes)
    piezas = [] if i0 == 0 else ["… "]
    for i, l in zip(t["ids"][i0:t["n_prompt"] + despues],
                    t["labels"][i0:t["n_prompt"] + despues]):
        txt = html.escape(TOK.decode([i])).replace("\n", "↵")
        fondo = "#dddddd" if l == -100 else "#a8e6b4"
        piezas.append(f'<span style="background:{fondo};color:#000;'
                      f'border:1px solid #fff;padding:0 1px">{txt}</span>')
    try:
        from IPython.display import HTML
        display(HTML("<div style='font-family:monospace;line-height:1.9'>"
                     + "".join(piezas) + "</div>"))
    except ImportError:
        pass
    n = t["n_prompt"]
    filas = [{"posición": j, "token": repr(TOK.decode([t["ids"][j]])),
              "id": t["ids"][j], "label": t["labels"][j],
              "¿cuenta en la loss?": "sí" if t["labels"][j] != -100 else "no"}
             for j in range(max(0, n - 4), min(len(t["ids"]), n + 4))]
    display(pd.DataFrame(filas).set_index("posición"))
    print(f"{len(t['ids'])} tokens: {n} de prompt (no cuentan) y "
          f"{len(t['ids']) - n} de completion (sí cuentan)")


ver_mascara(TOKS["train"][0])

In [ ]:
def lote(items):
    """Relleno a la derecha: pad en ids, 0 en la máscara, -100 en labels."""
    largo = max(len(t["ids"]) for t in items)
    ids = torch.full((len(items), largo), TOK.pad_token_id)
    labels = torch.full((len(items), largo), -100)
    mascara = torch.zeros((len(items), largo), dtype=torch.long)
    for i, t in enumerate(items):
        n = len(t["ids"])
        ids[i, :n], labels[i, :n] = torch.tensor(t["ids"]), torch.tensor(t["labels"])
        mascara[i, :n] = 1
    return {"ids": ids.to(DEVICE), "mascara": mascara.to(DEVICE),
            "labels": labels.to(DEVICE)}


def perdida_suma(b):
    """(suma de −log p del token correcto, número de tokens que cuentan).
    El logit de la posición i predice el token i + 1: de ahí el corrimiento."""
    logits = LM(input_ids=b["ids"], attention_mask=b["mascara"]).logits
    objetivo = b["labels"][:, 1:]
    suma = F.cross_entropy(logits[:, :-1].float().flatten(0, 1),
                           objetivo.flatten(), ignore_index=-100,
                           reduction="sum")
    return suma, (objetivo != -100).sum()


@torch.no_grad()
def perdida_val(parte="validation", lora=True):
    """Loss media por token de completion en los primeros n_val_loss."""
    items, suma, cuenta = TOKS[parte][:P["n_val_loss"]], 0.0, 0
    estaba = LM.training
    LM.eval()
    with _sin_adaptador(LM, not lora):
        for i in range(0, len(items), 2 * P["batch"]):
            with torch.autocast("cuda", dtype=torch.float16, enabled=CUDA):
                s, n = perdida_suma(lote(items[i:i + 2 * P["batch"]]))
            suma, cuenta = suma + s.item(), cuenta + n.item()
    LM.train(estaba)
    return suma / max(cuenta, 1)

1. Toma `P["acum"]` batches de train y rellena a la derecha. La máscara de atención marca el *padding* y su label `-100` lo excluye de la loss.
2. Hace el forward y el backward por batch. Divide cada suma de loss por el total de tokens evaluados de **todos** los batches de ese paso. Así los gradientes acumulados corresponden al promedio por token del paso completo.
3. Aplica *gradient clipping* para limitar la norma del gradiente y actualiza $A$ y $B$. Los pesos base siguen congelados.

| Técnica | Para qué se usa |
|---|---|
| AdamW | Actualiza el adaptador con el learning rate de `P["lr"]`. |
| Warmup y cosine schedule | Sube el learning rate al inicio y luego lo reduce siguiendo una curva coseno. |
| Mixed precision en GPU | Usa `autocast` y `GradScaler` para reducir el riesgo de perder gradientes pequeños. Los pesos de $A$ y $B$ se mantienen en fp32. |
| Gradient checkpointing | Ahorra memoria al recalcular activaciones durante el backward. |
| LoRA dropout | Enmascara al azar parte de la entrada a la rama LoRA durante el entrenamiento. |

`entrenar()` crea un adaptador nuevo cada vez. Si `GradScaler` detecta gradientes no finitos, se salta la actualización y el scheduler no avanza. Si se agotan los intentos, la celda avisa cuántas actualizaciones válidas hizo.

In [ ]:
def _preparar():
    """Adaptador nuevo, checkpointing, AdamW, cosine con warmup, GradScaler."""
    lora_nuevo()
    liberar()
    if CUDA:
        torch.cuda.reset_peak_memory_stats()
    LM.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False})
    LM.enable_input_require_grads()
    LM.config.use_cache = False           # el caché no sirve al entrenar
    entrenables = [p for p in LM.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(entrenables, lr=P["lr"], weight_decay=0.0)
    sched = get_cosine_schedule_with_warmup(
        opt, num_warmup_steps=max(1, round(0.05 * P["pasos"])),
        num_training_steps=P["pasos"])
    return entrenables, opt, sched, torch.amp.GradScaler("cuda", enabled=CUDA)


def _batches(semilla_=0):
    """Batches de train sin fin: cada época, un orden nuevo (y fijo)."""
    g, train = np.random.default_rng(semilla_), TOKS["train"]
    n = min(P["batch"], len(train))
    while True:
        orden = g.permutation(len(train))
        for i in range(0, len(orden) - n + 1, n):     # batches completos
            yield lote([train[j] for j in orden[i:i + n]])

In [ ]:
def entrenar():
    """Un LoRA desde cero. Devuelve el historial de loss train/validation."""
    semilla(0)
    entrenables, opt, sched, escalador = _preparar()
    hist = {"train": [], "validation": [(0, perdida_val())], "seg": []}
    fuente, paso, intentos = _batches(), 0, 0
    LM.train()
    while paso < P["pasos"] and intentos < 2 * P["pasos"]:
        intentos += 1
        t0 = time.perf_counter()
        lotes = [next(fuente) for _ in range(P["acum"])]
        n_tok = sum((b["labels"][:, 1:] != -100).sum() for b in lotes)
        opt.zero_grad(set_to_none=True)
        total = 0.0
        for b in lotes:                   # acumula acum gradientes
            with torch.autocast("cuda", dtype=torch.float16, enabled=CUDA):
                s, _ = perdida_suma(b)
            perdida = s / n_tok           # media sobre TODOS los tokens del paso
            escalador.scale(perdida).backward()
            total += perdida.item()
        escalador.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(entrenables, 1.0)
        escala_previa = escalador.get_scale()
        escalador.step(opt)
        escalador.update()
        if escalador.get_scale() < escala_previa:   # inf/NaN: paso saltado
            continue
        paso += 1
        sched.step()
        if CUDA:
            torch.cuda.synchronize()
        hist["seg"].append(time.perf_counter() - t0)
        hist["train"].append((paso, total))
        if paso % P["val_cada"] == 0 or paso == P["pasos"]:
            hist["validation"].append((paso, perdida_val()))
    if paso < P["pasos"]:
        print(f"¡Ojo! solo {paso} pasos válidos de {P['pasos']} (inf/NaN)")
    LM.eval()
    LM.config.use_cache = True            # para generar rápido
    return hist

In [ ]:
inicio = time.perf_counter()
HIST = entrenar()
seg = HIST["seg"][3:] or HIST["seg"]
print(f"✓ {len(HIST['train'])} pasos en {time.perf_counter() - inicio:.0f} s "
      f"({np.mean(seg):.2f} s/paso)")
print(f"loss de validation: {HIST['validation'][0][1]:.3f} (base) → "
      f"{HIST['validation'][-1][1]:.3f} (LoRA)")
if CUDA:
    print(f"memoria pico de la GPU: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

In [ ]:
def curvas(hist, titulo="Loss solo en el completion"):
    """Train delgada y tenue (un batch por punto); validation gruesa."""
    fig, eje = plt.subplots(figsize=(7.5, 3.6))
    eje.plot(*zip(*hist["train"]), color=COLORES["train"], lw=0.8,
             alpha=0.5, label="train (cada paso)")
    eje.plot(*zip(*hist["validation"]), color=COLORES["validation"], lw=2.5,
             marker="o", ms=4, label="validation")
    eje.set(xlabel="paso", ylabel="loss por token (nats)", title=titulo)
    eje.legend()
    plt.tight_layout()
    plt.show()


curvas(HIST)

**Cómo leer la gráfica.** Sigue la tendencia azul de **train**, que registra la loss de los batches de cada paso. En naranja, **validation** vuelve a medir los mismos ejemplos tokenizados. Su punto inicial corresponde a la salida del base, porque $B=0$.

La loss está en *nats* por token. Cuanto menor es, mayor es la media geométrica de las probabilidades asignadas a los tokens esperados, $e^{-\mathcal L}$. Si train baja y validation sube de forma sostenida, puede haber sobreajuste. La mejora puede ser pequeña si el base ya resuelve la tarea o si los ejemplos muestran patrones difíciles de aprender con esta configuración. Contrasta las curvas con las respuestas de la sección 07.

## 06 · Ver LoRA por dentro

Para inspeccionar el adaptador entrenado, la celda construye $\Delta W=\frac{\alpha}{r}BA$ en cada capa adaptada. El espectro muestra cuántas direcciones usa esa corrección. La norma relativa dice cuánto pesa frente a $W_0$. La perilla $\lambda$ y el merge permiten comprobar cómo se aplica al generar.

In [ ]:
def delta_w(capa):
    """ΔW = (α/r)·B·A de una capa LoRA, en CPU y fp32: [d_salida, d_entrada]."""
    A = capa.lora_A["default"].weight.detach().float().cpu()
    B = capa.lora_B["default"].weight.detach().float().cpu()
    return capa.scaling["default"] * (B @ A)


filas = []
for n, m in capas_lora().items():
    dw, w0 = delta_w(m), m.base_layer.weight.detach().float().cpu()
    filas.append({"nombre": n, "bloque": int(n.split(".layers.")[1].split(".")[0]),
                  "lineal": n.split(".")[-1],
                  "relativa": (dw.norm() / w0.norm()).item()})
NORMAS = pd.DataFrame(filas)
mayor = NORMAS.loc[NORMAS["relativa"].idxmax()]
print(f"La capa que más cambió: {mayor['nombre']} "
      f"(‖ΔW‖/‖W₀‖ = {mayor['relativa']:.4f})")

La SVD descompone una matriz en direcciones ponderadas por sus valores singulares $\sigma_1\ge\sigma_2\ge\dots\ge 0$. El rango exacto cuenta los valores singulares no nulos. Como $\Delta W$ pasa por un espacio de dimensión $r$, puede usar hasta $r$ direcciones.

**Cómo leer la gráfica.** En la capa con mayor cambio relativo, compara $W_0$ (gris) y $\Delta W$ (naranja). Cada curva divide sus valores singulares por su propio $\sigma_1$ y usa escala logarítmica. Si hay valores de $\Delta W$ a la derecha de la línea que marca el corte en $r$, deben quedar cerca de cero. La caída puede empezar antes si se usan menos direcciones. El redondeo deja residuos pequeños y, si $\Delta W=0$, el cociente queda indefinido.

In [ ]:
# (a) Valores singulares de ΔW en la capa que más cambió, y de su W₀.
capa = capas_lora()[mayor["nombre"]]
s_dw = torch.linalg.svdvals(delta_w(capa))
s_w0 = torch.linalg.svdvals(capa.base_layer.weight.detach().float().cpu())
r, k = P["rango"], min(3 * P["rango"], len(s_dw))
fig, eje = plt.subplots(figsize=(7.5, 3.4))
eje.semilogy(range(1, k + 1), s_w0[:k] / s_w0[0], "o-", color="gray",
             ms=3, label="W₀ (congelada)")
eje.semilogy(range(1, k + 1), (s_dw[:k] / s_dw[0]).clamp_min(1e-12), "o-",
             color=COLORES["validation"], ms=4, label="ΔW (LoRA)")
eje.axvline(r + 0.5, color="black", ls="--", lw=1)
eje.text(r + 0.7, 1e-3, f"r = {r}", fontsize=9)
eje.set(xlabel="índice del valor singular", ylabel="σᵢ / σ₁ (escala log)",
        title=f"Valores singulares · {mayor['lineal']} del bloque "
              f"{mayor['bloque']}")
eje.legend()
plt.tight_layout()
plt.show()
if len(s_dw) > r:
    print(f"σ_{r + 1} / σ_1 de ΔW = {(s_dw[r] / s_dw[0]).item():.1e}  "
          f"(cero numérico: rank(ΔW) ≤ {r})")

La razón $\lVert\Delta W\rVert_F/\lVert W_0\rVert_F$ compara el tamaño de la corrección con el peso original. La norma de Frobenius $\lVert M\rVert_F=\sqrt{\sum_{i,j}M_{ij}^2}$ reúne las entradas de una matriz en una sola magnitud.

**Cómo leer la gráfica.** Cada grupo corresponde a un bloque del transformer y cada color a una proyección. Una barra alta indica un cambio grande para esa capa. Cuánto afecta a las respuestas depende también de cómo se propaga la corrección. La tabla promedia por tipo de proyección a través de los bloques.

In [ ]:
# (b) ‖ΔW‖_F / ‖W₀‖_F por bloque y por lineal.
tabla_n = NORMAS.pivot(index="bloque", columns="lineal", values="relativa")
tabla_n = tabla_n[[m for m in P["modulos"] if m in tabla_n.columns]]
ancho = 0.8 / len(tabla_n.columns)
fig, eje = plt.subplots(figsize=(10, 3.6))
for j, lineal in enumerate(tabla_n.columns):
    eje.bar(tabla_n.index + (j - (len(tabla_n.columns) - 1) / 2) * ancho,
            tabla_n[lineal], width=ancho, color=plt.cm.tab10(j), label=lineal)
eje.set(xlabel="bloque del transformer (0 = el más cercano a la entrada)",
        ylabel="‖ΔW‖ / ‖W₀‖", title="¿Qué capas cambió más tu LoRA?")
eje.legend(ncol=len(tabla_n.columns), fontsize=8)
plt.tight_layout()
plt.show()
display(tabla_n.mean().rename("promedio por lineal").to_frame().T.round(4))

Al generar, `escala_lora` aplica $W=W_0+\lambda\,\Delta W$ en cada capa adaptada y restaura la escala al salir.

| $\lambda$ | Qué aplica |
|---|---|
| $0$ | Corrección nula. La celda compara las respuestas con las del adaptador apagado. |
| $0.5$ | La mitad de la corrección entrenada. |
| $1$ | La corrección entrenada. |
| $1.5$ | Una vez y media la corrección entrenada. |

**Cómo leer la tabla.** En cada fila, compara el mismo prompt a distintas escalas. Se usa el primer prompt de `prompts.txt` y el primero de validation, ambos con greedy decoding. Una respuesta puede quedarse igual si se siguen eligiendo los mismos tokens, o cambiar mucho a partir del primer token que sale distinto.

In [ ]:
@contextlib.contextmanager
def escala_lora(lam):
    """Multiplica α/r de TODAS las capas LoRA por λ; al salir, lo restaura."""
    capas = list(capas_lora().values())
    antes = [dict(m.scaling) for m in capas]
    try:
        for m in capas:
            for nombre in m.scaling:
                m.scaling[nombre] *= lam
        yield
    finally:
        for m, a in zip(capas, antes):
            m.scaling.update(a)


# (c) La perilla: el mismo prompt, greedy (sin azar), cuatro valores de λ.
PROMPTS_L = [PROMPTS[0], DATOS["validation"][0]["prompt"]]
por_lambda = {}
for lam in [0, 0.5, 1, 1.5]:
    with escala_lora(lam):
        por_lambda[f"λ = {lam}"] = generar(PROMPTS_L)
mostrar(respuestas(PROMPTS_L, **por_lambda))
print("λ = 0 da lo mismo que el base:",
      por_lambda["λ = 0"] == generar(PROMPTS_L, lora=False))

`merge_and_unload()` suma $\Delta W$ a $W_0$ en una copia del modelo y elimina las capas LoRA de esa copia. La celda cuenta las capas y los parámetros restantes, compara logits y comprueba si coinciden las respuestas de `prompts.txt`.

En aritmética exacta, sumar los pesos o aplicar la corrección en el forward da el mismo resultado lineal. En precisión finita, las dos operaciones redondean en puntos distintos. Lee la diferencia máxima junto con la magnitud de los logits. Incluso una diferencia pequeña puede cambiar el token elegido si dos logits están muy cerca.

In [ ]:
# (d) merge_and_unload sobre una COPIA: W = W₀ + (α/r)BA en una sola matriz.
import copy

copia = copy.deepcopy(LM)
fusionado = copia.merge_and_unload()
print("capas LoRA que quedan tras fusionar:",
      sum(isinstance(m, LoraLayer) for m in fusionado.modules()))
print(f"parámetros: base {n_lm:,} · fusionado "
      f"{sum(p.numel() for p in fusionado.parameters()):,}")
TOK.padding_side = "left"
x = TOK([formatear(p) for p in PROMPTS], return_tensors="pt", padding=True,
        add_special_tokens=False).to(DEVICE)
TOK.padding_side = "right"
with torch.no_grad():
    con_adaptador = LM(**x).logits.float()
    con_fusion = fusionado(**x).logits.float()
dif = (con_adaptador - con_fusion).abs().max().item()
print(f"max |logits adaptador − logits fusionado| = {dif:.2e} "
      f"(escala de los logits: {con_adaptador.abs().max().item():.1f})")
iguales = generar(PROMPTS) == generar(PROMPTS, modelo=fusionado)
print("¿mismas respuestas greedy en prompts.txt?", iguales)
del copia, fusionado, con_adaptador, con_fusion
liberar()

## 07 · Métricas, explicadas

Compara **base y LoRA en validation**. La loss y la perplexity evalúan los completions de referencia que quedaron tras tokenizar y recortar. Exact match y token F1 comparan las respuestas generadas con las referencias completas. La celda imprime cuántos ejemplos usa en cada cálculo.

**Loss**

En cada posición evaluada, el modelo recibe el prompt y los tokens **de referencia** anteriores (*teacher forcing*). La loss $\mathcal L$ de la sección 05 promedia $-\log p$ para los tokens esperados.

| Qué mide | Rango | Qué es bueno | Trampa |
|---|---|---|---|
| Sorpresa media, en *nats* por token del completion, ante la respuesta esperada. | $[0,\infty)$ | Menor loss sobre los mismos tokens. | Al generar, el modelo sigue sus propios tokens. Una loss baja puede convivir con respuestas que se desvían pronto. |

Si el recorte deja el split sin tokens evaluables, la función devuelve loss cero por cómo protege la división. Revisa el conteo antes de interpretar esa cifra.

**Perplexity**

La perplexity es $e^{\mathcal L}$, el inverso de la media geométrica de las probabilidades asignadas a los tokens esperados.

| Qué mide | Rango | Qué es bueno | Trampa |
|---|---|---|---|
| La misma sorpresa expresada como número efectivo de opciones equiprobables por token. | $[1,\infty)$ en la definición. | Menor perplexity sobre los mismos tokens. | Ordena los modelos igual que la loss. Depende del tokenizer y de los datos. En cifras extremas la tabla limita el exponente para evitar desbordamientos. |

**Exact match normalizado**

| Qué mide | Rango | Qué es bueno | Trampa |
|---|---|---|---|
| Porcentaje de respuestas que coinciden por completo tras pasar a minúsculas y colapsar espacios. | De 0 a 100 %. | Un porcentaje alto cuando la tarea pide una forma de respuesta bien definida. | Una consulta equivalente con otro alias o un bloque de código extra puede valer cero. La puntuación y el orden siguen contando. |

**Token F1 (estilo SQuAD)**

Aquí los tokens son fragmentos separados por espacios tras la misma normalización. Artículos y puntuación se conservan. Sean $n_p$ y $n_r$ las longitudes de la predicción y la referencia, y $m$ el número de tokens que comparten contando repeticiones. Con ambas respuestas no vacías, la *precision* es $m/n_p$ y el *recall* es $m/n_r$. El F1 combina las dos en $F_1=2m/(n_p+n_r)$. Si una respuesta está vacía, el código devuelve cero. Si ambas lo están, devuelve uno.

| Qué mide | Rango | Qué es bueno | Trampa |
|---|---|---|---|
| Solapamiento de tokens, con crédito parcial. Baja si sobran tokens o faltan tokens esperados. | De 0 a 100 % en la tabla resumen. | Un F1 alto cuando importa recuperar el contenido de la referencia. | Ignora el orden. Una respuesta puede tener casi todos los tokens correctos y fallar en un operador que cambia el sentido. |

La celda ilustra la diferencia con `>` y `>=` en SQL. Exact match vale cero, mientras F1 da crédito por los tokens compartidos.

In [ ]:
from collections import Counter


def normalizar(texto):
    """Minúsculas y espacios colapsados (ni puntuación ni artículos se tocan)."""
    return " ".join(texto.lower().split())


def exact_match(pred, real):
    return float(normalizar(pred) == normalizar(real))


def token_f1(pred, real):
    """F1 entre las bolsas de palabras de pred y real (estilo SQuAD)."""
    p, r = normalizar(pred).split(), normalizar(real).split()
    if not p or not r:
        return float(p == r)
    comunes = sum((Counter(p) & Counter(r)).values())
    if comunes == 0:
        return 0.0
    precision, recall = comunes / len(p), comunes / len(r)
    return 2 * precision * recall / (precision + recall)


# Un ejemplo a mano, para ver la diferencia entre las dos:
real = "SELECT name FROM students WHERE age > 20"
pred = "select name from students where age >= 20"
print(f"exact match = {exact_match(pred, real):.0f} · "
      f"token F1 = {token_f1(pred, real):.3f}")

In [ ]:
def evaluar(parte):
    """Base y LoRA en los primeros n_eval ejemplos de `parte`: una tabla con
    las cuatro métricas y las respuestas de cada modelo."""
    ejs = DATOS[parte][:P["n_eval"]]
    reales = [e["completion"] for e in ejs]
    filas, preds = {}, {}
    for nombre, lora in [("base", False), ("LoRA", True)]:
        t0 = time.perf_counter()
        loss = perdida_val(parte, lora=lora)
        preds[nombre] = generar([e["prompt"] for e in ejs], lora=lora)
        filas[nombre] = {
            "loss": loss, "perplexity": math.exp(min(loss, 50)),
            "exact match %": 100 * np.mean(
                [exact_match(p, r) for p, r in zip(preds[nombre], reales)]),
            "token F1 %": 100 * np.mean(
                [token_f1(p, r) for p, r in zip(preds[nombre], reales)])}
        print(f"{nombre}: {time.perf_counter() - t0:.0f} s")
    print(f"({len(ejs)} ejemplos de {parte} para generar, "
          f"{min(len(TOKS[parte]), P['n_val_loss'])} para la loss)")
    return pd.DataFrame(filas).T, preds


def por_ejemplo(parte, preds, n=5):
    """Las primeras n respuestas, con su exact match y su F1."""
    ejs = DATOS[parte][:min(n, len(preds["base"]))]
    df = respuestas([e["prompt"] for e in ejs],
                    esperado=[e["completion"] for e in ejs],
                    base=preds["base"][:n], LoRA=preds["LoRA"][:n])
    df["EM base / LoRA"] = [
        f"{exact_match(b, e['completion']):.0f} / "
        f"{exact_match(l, e['completion']):.0f}"
        for b, l, e in zip(preds["base"], preds["LoRA"], ejs)]
    df["F1 base / LoRA"] = [
        f"{token_f1(b, e['completion']):.2f} / "
        f"{token_f1(l, e['completion']):.2f}"
        for b, l, e in zip(preds["base"], preds["LoRA"], ejs)]
    return df

In [ ]:
VAL_TABLA, VAL_PRED = evaluar("validation")
display(VAL_TABLA.round(3))
mostrar(por_ejemplo("validation", VAL_PRED))

**Cómo leer las tablas.** Compara base y LoRA por columna. Busca menos loss y perplexity y más exact match y F1, según lo que importe en tu tarea. Abajo, cada fila muestra el prompt, la referencia y ambas respuestas. Los scores por ejemplo van de cero a uno. Si F1 sube y exact match se queda igual, busca qué conserva la respuesta y qué diferencias aún importan. Si la salida se ve cortada en la tabla, recuerda que los scores usan el texto completo que devolvió el generador.

## 08 · Testeo, una sola vez

> **Ejecuta la evaluación de test una sola vez**, con la configuración que elegiste en validation. Si cambias el modelo tras ver los resultados de test, necesitarás otro conjunto reservado para evaluar el resultado final.

**Cómo leer las tablas.** La primera repite las cuatro métricas de la sección 07 en **test**. La segunda muestra ejemplos con la respuesta esperada, las dos generaciones y sus scores. Compara base y LoRA en el mismo split. Si el resultado difiere de validation, mira cuántos ejemplos entraron en cada medición y qué errores aparecen. Con muestras pequeñas, unas pocas respuestas pueden mover el promedio.

In [ ]:
# El test, por primera y única vez.
TEST_TABLA, TEST_PRED = evaluar("test")
display(TEST_TABLA.round(3))
mostrar(por_ejemplo("test", TEST_PRED))

**Tus prompts, antes y después.** La tabla recupera las respuestas del base a `prompts.txt` y genera nuevas con LoRA. Estos prompts son ejemplos exploratorios. Úsalos para ver si cambia el formato, si el modelo sigue la instrucción y si el contenido mejora.

In [ ]:
# Tus prompts de prueba: el base de §03 contra tu LoRA.
mostrar(respuestas(PROMPTS, base=BASE_PROMPTS, LoRA=generar(PROMPTS)))

**Olvido con el adaptador activo**

| Qué mide | Rango | Qué es bueno | Trampa |
|---|---|---|---|
| Si al activar LoRA se deterioran respuestas en tres tareas generales. | Comparación cualitativa, sin escala numérica. | Mantener el contenido correcto y seguir la instrucción al menos tan bien como el base. | Tres preguntas cubren poco. Una diferencia de estilo puede ser inocua y el base también puede equivocarse. |

**Cómo leer la tabla.** Revisa si se conserva el dato pedido, si la explicación sigue siendo correcta y si se respeta el formato solicitado. Si LoRA impone el formato de tu dataset en otras tareas, anota el fallo de instrucción. Esta prueba compara el comportamiento con el adaptador activo. Al apagarlo, vuelves a usar los pesos base.

In [ ]:
# Tres preguntas que no tienen nada que ver con tu dataset.
AJENAS = ["¿Cuál es la capital de Francia? Contesta en una frase.",
          "Explica en dos frases qué es la fotosíntesis.",
          "Escribe un haiku sobre el mar."]
mostrar(respuestas(AJENAS, base=generar(AJENAS, lora=False, max_new=96),
                   LoRA=generar(AJENAS, max_new=96)))

La celda guarda los pesos del adaptador y su configuración en `mi_lora_texto`. Añade un `LEEME.md` con el modelo base, la configuración, el número de actualizaciones válidas, la ficha del dataset y las métricas de test. Empaqueta la carpeta en `mi_lora_texto.zip` e inicia la descarga en Colab.

Si el navegador bloquea la descarga, abre el panel **📁 Archivos** y elige «Descargar» sobre el zip. Fuera de Colab, el zip queda en la carpeta actual. Para reutilizarlo, descomprime el zip y carga el adaptador con PEFT sobre el mismo modelo base que figura en la ficha.

In [ ]:
# Solo A y B (y su configuración) a una carpeta, y la carpeta a un zip.
import shutil

CARPETA = "mi_lora_texto"
LM.save_pretrained(CARPETA)
with open(os.path.join(CARPETA, "LEEME.md"), "w", encoding="utf-8") as f:
    f.write(f"# Mi LoRA de texto\n\n- Modelo base: {MODELO}\n"
            f"- r = {P['rango']}, α = {P['alfa']}, capas: {P['modulos']}\n"
            f"- {len(HIST['train'])} pasos, lr = {P['lr']}\n\n"
            f"## Mi dataset\n\n{LEEME}\n\n## Test (una vez)\n\n"
            f"{TEST_TABLA.round(3).to_string()}\n")
zip_lora = shutil.make_archive(CARPETA, "zip", CARPETA)
print(f"{zip_lora}: {os.path.getsize(zip_lora) / 1e6:.1f} MB")
try:
    from google.colab import files
    files.download(zip_lora)              # baja a tu computadora
except ImportError:
    print("(fuera de Colab: el zip queda en la carpeta actual)")

Usa esta receta **en validation, antes de abrir test**. Cambia una cosa a la vez y vuelve a ejecutar desde la sección donde se configura.

| Lo que ves | Qué probar |
|---|---|
| Las respuestas apenas cambian y todavía fallan en la tarea. | Revisa si los completions muestran un patrón claro. Prueba más pasos o el modo largo si tienes memoria suficiente. |
| Train baja y validation sube de forma sostenida. | Prueba menos pasos o añade ejemplos variados. |
| Aparecen avisos de gradientes no finitos. | Prueba un learning rate menor y comprueba cuántas actualizaciones se completan. |
| Hay muchos recortes o descartes. | Acorta los ejemplos o aumenta `P["max_len"]` si cabe en memoria. Revisa si se conserva el final de la respuesta. |
| La generación se queda a medias o se repite hasta el límite. | Si falta espacio para terminar, sube `P["max_new"]`. Si hay repeticiones, revisa los finales de tus completions y los recortes de la sección 05. |
| F1 es alto y exact match es bajo. | Lee las respuestas. Distingue cambios de formato aceptables de errores que alteran el sentido. |
| Las respuestas de validation pierden detalles al aplicar LoRA. | Usa la perilla de la sección 06 para explorar una corrección más débil o prueba menos pasos. |
| Se agota la memoria al entrenar. | Reduce `P["batch"]`. Puedes aumentar `P["acum"]` para acercarte al mismo número de ejemplos por actualización. |

Conserva el zip junto con la comparación de test. Al presentar el resultado, acompaña las métricas con una respuesta que ilustre qué cambió al activar LoRA y qué le sigue costando al modelo.